In [ ]:
%%capture
!pip install keras-cv

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow" # other options: tensorflow or torch
import json
import keras_cv
import keras
from keras import ops
import tensorflow as tf

import cv2
import pandas as pd
import numpy as np
from glob import glob
from tqdm.notebook import tqdm
import joblib

import matplotlib.pyplot as plt

In [ ]:
class CFG:
    verbose = 1  # Verbosity
    seed = 47  # Random seed
    neg_sample = 0.01 # Downsample negative calss
    pos_sample = 5.0  # Upsample positive class
    preset = "efficientnetv2_b2_imagenet"  # Name of pretrained classifier
    image_size = [128, 128]  # Input image size
    epochs = 10 # Training epochs
    batch_size = 256  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    class_names = ['target']
    num_classes = 1

keras.utils.set_random_seed(CFG.seed)

In [ ]:
ROOT = str(os.getcwd())
BASE_PATH = f"{ROOT}/drive/MyDrive/WAIT"
DATA_PATH = f"{BASE_PATH}/data/new"
MODEL_PATH = f"{BASE_PATH}/models/enhanced/"
PROC_PATH = f"{MODEL_PATH}/enhanced/preprocessed_dataset/"

In [ ]:
# Train + Valid
df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_train_metadata_clean.parquet')

# Testing
testing_df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_test_metadata_clean.parquet')

In [ ]:
print("Class Distribution Before Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

# Sampling
positive_df = df.query("target==0").sample(frac=CFG.neg_sample, random_state=CFG.seed)
negative_df = df.query("target==1").sample(frac=CFG.pos_sample, replace=True, random_state=CFG.seed)
df = pd.concat([positive_df, negative_df], axis=0).sample(frac=1.0)

print("\nCalss Distribution After Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(df['target']), y=df['target'])
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

In [ ]:
import h5py

training_validation_hdf5 = h5py.File(f"{DATA_PATH}/train-image.hdf5", 'r')
testing_hdf5 = h5py.File(f"{DATA_PATH}/test-image.hdf5", 'r')

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True) # ensure continuous index
df["fold"] = -1
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
for i, (training_idx, validation_idx) in enumerate(sgkf.split(df, y=df.target, groups=df.patient_id)):
    df.loc[validation_idx, "fold"] = int(i)

# Use first fold for training and validation
training_df = df.query("fold!=0")
validation_df = df.query("fold==0")
print(f"# Num Train: {len(training_df)} | Num Valid: {len(validation_df)}")

In [ ]:
# class distribution in training and validation
print("training:", training_df.target.value_counts(),
      "\nvalidation:", validation_df.target.value_counts(), sep='\n')

# 📊 | Tabular Features

### Enhanced


In [ ]:
# Categorical features which will be one hot encoded
CATEGORICAL_COLUMNS = ["sex", "anatom_site_general", "tbp_tile_type",
                       "tbp_lv_location", "attribution"]

NEW_CATEGORICAL_COLUMNS =["combined_anatomical_site"]

# Numerical features which will be normalized
NUMERIC_COLUMNS = ["age_approx", "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
           "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio", "tbp_lv_color_std_mean",
           "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM"]

NEW_NUMERIC_COLUMNS = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
         "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency", "color_consistency",

        "size_age_interaction", "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation", "overall_color_difference",
        "symmetry_perimeter_interaction", "comprehensive_lesion_index"]

# Tabular feature columns
FEAT_COLS = (CATEGORICAL_COLUMNS + NEW_CATEGORICAL_COLUMNS) + (NUMERIC_COLUMNS + NEW_NUMERIC_COLUMNS)

In [ ]:
def build_augmenter():
    # Define augmentations
    aug_layers = [
        keras_cv.layers.RandomCutout(height_factor=(0.02, 0.06), width_factor=(0.02, 0.06)),
        keras_cv.layers.RandomFlip(mode="horizontal"),
    ]

    # Apply augmentations to random samples
    aug_layers = [keras_cv.layers.RandomApply(x, rate=0.5) for x in aug_layers]

    # Build augmentation layer
    augmenter = keras_cv.layers.Augmenter(aug_layers)

    # Apply augmentations
    def augment(inp, label):
        images = inp["images"]
        aug_data = {"images": images}
        aug_data = augmenter(aug_data)
        inp["images"] = aug_data["images"]
        return inp, label
    return augment


def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        # Read jpeg image
        file_bytes = inp["images"]
        image = tf.io.decode_jpeg(file_bytes)

        # Resize
        image = tf.image.resize(image, size=target_size, method="area")

        # Rescale image
        image = tf.cast(image, tf.float32)
        image /= 255.0

        # Reshape
        image = tf.reshape(image, [*target_size, 3])

        inp["images"] = image
        return inp

    def decode_label(label, num_classes):
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [num_classes])
        return label

    def decode_with_labels(inp, label=None):
        inp = decode_image(inp)
        label = decode_label(label, CFG.num_classes)
        return (inp, label)

    return decode_with_labels if with_labels else decode_image


def build_dataset(
    isic_ids,
    hdf5,
    features,
    labels=None,
    batch_size=32,
    decode_fn=None,
    augment_fn=None,
    augment=False,
    shuffle=1024,
    cache=True,
    drop_remainder=False,
):
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)

    if augment_fn is None:
        augment_fn = build_augmenter()

    AUTO = tf.data.experimental.AUTOTUNE

    images = [None]*len(isic_ids)
    for i, isic_id in enumerate(tqdm(isic_ids, desc="Loading Images ")):
        images[i] = hdf5[isic_id][()]

    inp = {"images": images, "features": features}
    slices = (inp, labels) if labels is not None else inp

    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.cache() if cache else ds
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)
    if shuffle:
        ds = ds.shuffle(shuffle, seed=CFG.seed)
        opt = tf.data.Options()
        opt.deterministic = False
        ds = ds.with_options(opt)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder)
    ds = ds.map(augment_fn, num_parallel_calls=AUTO) if augment else ds
    ds = ds.prefetch(AUTO)
    return ds

In [ ]:
# Build Training & Validation Dataset

print("# Training:")
training_features = dict(training_df[FEAT_COLS])
training_ids = training_df.isic_id.values
training_labels = training_df.target.values
training_ds = build_dataset(training_ids, training_validation_hdf5, training_features,
                         training_labels, batch_size=CFG.batch_size,
                         shuffle=True, augment=True)

print("# Validation:")
validation_features = dict(validation_df[FEAT_COLS])
validation_ids = validation_df.isic_id.values
validation_labels = validation_df.target.values
validation_ds = build_dataset(validation_ids, training_validation_hdf5, validation_features,
                         validation_labels, batch_size=CFG.batch_size,
                         shuffle=False, augment=False)


In [ ]:
feature_space = keras.utils.FeatureSpace(
    features={
        # Categorical features encoded as integers
        "sex": "string_categorical",
        "anatom_site_general": "string_categorical",
        "tbp_tile_type": "string_categorical",
        "tbp_lv_location": "string_categorical",
        "combined_anatomical_site": "string_categorical",
        "attribution": "string_categorical",
        # Numerical features to discretize
        "age_approx": "float_discretized",
        # Numerical features to normalize
        "tbp_lv_nevi_confidence": "float_normalized",
        "clin_size_long_diam_mm": "float_normalized",
        "tbp_lv_areaMM2": "float_normalized",
        "tbp_lv_area_perim_ratio": "float_normalized",
        "tbp_lv_color_std_mean": "float_normalized",
        "tbp_lv_deltaLBnorm": "float_normalized",
        "tbp_lv_minorAxisMM": "float_normalized",
        "lesion_size_ratio" : "float_normalized",
        "lesion_shape_index" : "float_normalized",
        "hue_contrast" : "float_normalized",
        "luminance_contrast": "float_normalized",
        "lesion_color_difference": "float_normalized",
        "border_complexity": "float_normalized",
        "3d_position_distance": "float_normalized",
        "perimeter_to_area_ratio": "float_normalized",
        "lesion_visibility_score": "float_normalized",
        "symmetry_border_consistency": "float_normalized",
        "color_consistency": "float_normalized",
        "size_age_interaction": "float_normalized",
        "hue_color_std_interaction": "float_normalized",
        "lesion_severity_index": "float_normalized",
        "shape_complexity_index": "float_normalized",
        "color_contrast_index": "float_normalized",
        "log_lesion_area": "float_normalized",
        "mean_hue_difference": "float_normalized",
        "std_dev_contrast": "float_normalized",
        "color_shape_composite_index": "float_normalized",
        "3d_lesion_orientation": "float_normalized",
        "overall_color_difference": "float_normalized",
        "symmetry_perimeter_interaction": "float_normalized",
        "comprehensive_lesion_index": "float_normalized",
        # numerical features already normalized
        "normalized_lesion_size": "float",
        },
    output_mode="concat",
)

In [ ]:
training_ds_with_no_labels = training_ds.map(lambda x, _: x["features"])
feature_space.adapt(training_ds_with_no_labels)
print(feature_space.get_encoded_features().shape)

In [ ]:
# fs_dir = os.path.join(MODEL_PATH, "model-metadata")
# os.makedirs(fs_dir, exist_ok=True)
# filepath = os.path.join(fs_dir, "feature_space_test.keras")

# feature_space.save(filepath)
# print(f"FeatureSpace saved at: '{filepath}'")

In [ ]:
# Apply Feature Processing

training_ds = training_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

validation_ds = validation_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

# Testing
print("# Testing:")
testing_features = dict(testing_df[FEAT_COLS])
testing_ids = testing_df.isic_id.values
testing_ds = build_dataset(testing_ids, testing_hdf5,
                        testing_features, batch_size=CFG.batch_size,
                         shuffle=False, augment=False, cache=False)

# Apply feature space processing
testing_ds = testing_ds.map(
    lambda x: {"images": x["images"],
               "features": feature_space(x["features"])}, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
# loss & metric
auc = keras.metrics.AUC()
loss = keras.losses.BinaryCrossentropy(label_smoothing=0.02)

In [ ]:
# Define input layers
image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
inp = {"images":image_input, "features":feat_input}

# Branch for image input
backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
x1 = backbone(image_input)
x1 = keras.layers.GlobalAveragePooling2D()(x1)
x1 = keras.layers.Dropout(0.2)(x1)

# Branch for tabular/feature input
x2 = keras.layers.Dense(96, activation="selu")(feat_input)
x2 = keras.layers.Dense(128, activation="selu")(x2)
x2 = keras.layers.Dropout(0.1)(x2)

# Concatenate both branches
concat = keras.layers.Concatenate()([x1, x2])

# Output layer
out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

# Build model
model = keras.models.Model(inp, out)

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=loss,
    metrics=[auc],
)

# Model Summary
model.summary()

In [ ]:
# keras.utils.plot_model(model, show_shapes=True, show_layer_names=True, dpi=60)

In [ ]:
import math

def get_lr_callback(batch_size=8, mode='cos', epochs=10, plot=False):
    lr_start, lr_max, lr_min = 2.5e-5, 5e-6 * batch_size, 0.8e-5
    lr_ramp_ep, lr_sus_ep, lr_decay = 3, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if epoch < lr_ramp_ep: lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep: lr = lr_max
        elif mode == 'exp': lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
        elif mode == 'step': lr = lr_max * lr_decay**((epoch - lr_ramp_ep - lr_sus_ep) // 2)
        elif mode == 'cos':
            decay_total_epochs, decay_epoch_index = epochs - lr_ramp_ep - lr_sus_ep + 3, epoch - lr_ramp_ep - lr_sus_ep
            phase = math.pi * decay_epoch_index / decay_total_epochs
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:  # Plot lr curve if plot is True
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(epoch) for epoch in np.arange(epochs)], marker='o')
        plt.xlabel('epoch'); plt.ylabel('lr')
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=False)  # Create lr callback

lr_cb = get_lr_callback(CFG.batch_size, mode="exp", plot=False)

In [ ]:
# raise Exception("Stop here")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LORE (LOcal Rule Extractor) — Tabular XAI Callback
# Explains predictions via local IF-THEN rules from a surrogate decision tree.
# Two modes per epoch: 'context' (mean image fixed) and 'tabular_only' (zero image).
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
%%capture
!git clone https://github.com/kdd-lab/XAI-Lib.git /content/XAI-Lib
!python -m pip install -e /content/XAI-Lib

In [ ]:
import sys
sys.path.insert(0, "/content/XAI-Lib/src")

In [ ]:
_ = feature_space.get_encoded_features()

# Get the feature order as used by FeatureSpace
try:
    feature_order = list(feature_space.features.keys())
except AttributeError:
    # Fallback: you know the order from how you created the FeatureSpace
    feature_order = [
        "sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location",
        "combined_anatomical_site", "attribution",
        "age_approx",
        "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
        "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio",
        "tbp_lv_color_std_mean", "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM",
        "lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
        "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency",
        "color_consistency", "size_age_interaction",
        "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation",
        "overall_color_difference", "symmetry_perimeter_interaction",
        "comprehensive_lesion_index",
        "normalized_lesion_size",
    ]

In [ ]:
from keras.layers import StringLookup, Discretization, Normalization

encoded_feature_names = []
feature_dim_sizes = {}

for name in feature_order:
    preproc = feature_space.preprocessors[name]

    # 1) Categorical features: StringLookup → one-hot of size vocabulary_size
    if isinstance(preproc, StringLookup):
        vocab = preproc.get_vocabulary()
        # each token gets its own encoded dim
        for token in vocab:
            encoded_feature_names.append(f"{name}__{token}")
        d = len(vocab)

    # 2) Discretized float (age_approx): Discretization → one-hot over bins
    elif isinstance(preproc, Discretization):
        boundaries = preproc.get_config().get("bin_boundaries", [])
        n_bins = len(boundaries) + 1
        for b in range(n_bins):
            encoded_feature_names.append(f"{name}_bin_{b}")
        d = n_bins

    # 3) Normalized numeric or raw float: 1 dim
    elif isinstance(preproc, Normalization):
        encoded_feature_names.append(name)
        d = 1

    else:
        # "float" (already normalized) or any other scalar
        encoded_feature_names.append(name)
        d = 1

    feature_dim_sizes[name] = d

# sanity check
print("per-feature dim sizes:", feature_dim_sizes)
print("len(encoded_feature_names):", len(encoded_feature_names))
print("feature_space encoded dim:", feature_space.get_encoded_features().shape[-1])

len(encoded_feature_names) == feature_space.get_encoded_features().shape[-1] == 128

In [ ]:
def aggregate_raw_importance(encoded_feature_names, mean_abs_vec):
    agg = {}
    for j, val in enumerate(mean_abs_vec):
        name = encoded_feature_names[j]
        if "__" in name:
            raw = name.split("__", 1)[0]
        elif name.startswith("age_approx_bin_"):
            raw = "age_approx"
        else:
            raw = name
        agg.setdefault(raw, 0.0)
        agg[raw] += float(abs(val))
    return agg

In [ ]:
# ── LORE helpers ──
# (No translation needed — LORE now operates on raw features directly)

In [ ]:
import seaborn as sns

def plot_topk_grouped_raw_global_pos_neg(
    global_imp, pos_imp, neg_imp, k=10, epoch_num=None, figsize=(10, 5), out_path=None
):
    # Ranking
    items = sorted(global_imp.items(), key=lambda kv: kv[1], reverse=True)[:k]
    names = [k for k, _ in items]

    df = pd.DataFrame({
        "feature": names,
        "Global":   [global_imp.get(n, 0.0) for n in names],
        "Pos preds":[pos_imp.get(n,   0.0) for n in names],
        "Neg preds":[neg_imp.get(n,   0.0) for n in names],
    })

    df_melt = df.melt(id_vars="feature", var_name="Class", value_name="Importance")

    plt.figure(figsize=figsize)
    ax = sns.barplot(
        data=df_melt,
        y="feature", x="Importance", hue="Class",
        palette=(
            sns.color_palette("bright")[4],  # Global
            sns.color_palette("bright")[0],  # Pos
            sns.color_palette("bright")[3],  # Neg
        ),
        edgecolor="black",
    )

    for container in ax.containers:
        ax.bar_label(container, fmt="{:.2f}", padding=4, fontsize=7)

    plt.ylabel("Mean |SHAP|")
    plt.xlabel("")
    title = f"Feature Importance (top {k}) – Global vs Pos vs Neg"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    plt.title(title)
    plt.tight_layout()

    if out_path:
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
def encoded_indices_for_raw(encoded_feature_names, feat_name):
    idxs = []
    for j, name in enumerate(encoded_feature_names):
        if "__" in name:
            if name.split("__", 1)[0] == feat_name:
                idxs.append(j)
        elif name.startswith(f"{feat_name}_bin_"):
            idxs.append(j)
        elif name == feat_name:
            idxs.append(j)
    return np.array(idxs, dtype=int)


def select_top_numeric_features(global_importance_raw, raw_batch_df, k=4):
    """Return top-k raw features among those that are numeric-only."""
    numeric_feats = [
        f for f in global_importance_raw.keys()
        if f in raw_batch_df.columns and raw_batch_df[f].dtype.kind in "if"  # int or float
    ]

    ordered = sorted(
        numeric_feats,
        key=lambda name: global_importance_raw[name],
        reverse=True,
    )

    return ordered[:k]

In [ ]:
def plot_shap_scatter_top4_numeric(
    shap_vals,
    preds,
    encoded_feature_names,
    global_importance_raw,
    raw_batch_df,
    explained_indices,
    epoch_num=None,
    positive_threshold=0.5,
    out_path=None,
):
    top_feats = select_top_numeric_features(global_importance_raw, raw_batch_df, k=4)

    pred_scores = preds[:, 0]
    pos_mask = pred_scores >= positive_threshold
    neg_mask = ~pos_mask

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes = axes.ravel()

    title = "SHAP values vs. top numeric features (pos vs. neg predictions)"
    if epoch_num is not None:
        title += f" – {epoch_num}"
    fig.suptitle(title, fontsize=14)

    colors = sns.color_palette("bright", 5)
    pos_color, neg_color = colors[0], colors[3]

    for i, (ax, feat_name) in enumerate(zip(axes, top_feats)):
        enc_idx = encoded_indices_for_raw(encoded_feature_names, feat_name)
        if enc_idx.size == 0:
            ax.set_title(f"{feat_name} (no encoded dims)")
            ax.axis("off")
            continue

        shap_feat = shap_vals[:, enc_idx].sum(axis=1)

        raw_vals_batch = raw_batch_df[feat_name].values
        raw_vals = raw_vals_batch[explained_indices]

        x_pos = raw_vals[pos_mask]
        y_pos = shap_feat[pos_mask]
        x_neg = raw_vals[neg_mask]
        y_neg = shap_feat[neg_mask]

        ax.scatter(
            x_pos, y_pos,
            s=18, marker="o", alpha=0.7,
            label="Positive", color=pos_color, edgecolors="none",
        )
        ax.scatter(
            x_neg, y_neg,
            s=18, marker="o", alpha=0.8,
            label="Negative", color=neg_color, edgecolors="none",
        )

        ax.axhline(0.0, color="grey", linewidth=1, linestyle="--", alpha=0.7)
        ax.set_title(feat_name)

        if i in (0, 2):
            ax.set_ylabel("SHAP value")

    for i in range(len(top_feats), 4):
        axes[i].axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper right")

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    if out_path is not None:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
import seaborn as sns

def plot_lore_feature_frequency(
    freq_all: dict, freq_pos: dict, freq_neg: dict,
    k: int = 10, epoch_num=None, mode: str = '', figsize=(10, 5), out_path=None,
):
    """Barplot of rule-feature frequencies: Global / Pos / Neg predictions."""
    items = sorted(freq_all.items(), key=lambda kv: kv[1], reverse=True)[:k]
    if not items:
        return
    names = [n for n, _ in items]
    df = pd.DataFrame({
        'feature': names,
        'Global': [freq_all.get(n, 0) for n in names],
        'Pos preds': [freq_pos.get(n, 0) for n in names],
        'Neg preds': [freq_neg.get(n, 0) for n in names],
    })
    df_melt = df.melt(id_vars='feature', var_name='Class', value_name='Rule frequency')
    plt.figure(figsize=figsize)
    ax = sns.barplot(
        data=df_melt, y='feature', x='Rule frequency', hue='Class',
        palette=(sns.color_palette('bright')[4], sns.color_palette('bright')[0], sns.color_palette('bright')[3]),
        edgecolor='black',
    )
    for container in ax.containers:
        ax.bar_label(container, fmt='%d', padding=4, fontsize=7)
    title = f'LORE Feature Frequency in Rules (top {k})'
    if mode:
        title += f' [{mode}]'
    if epoch_num is not None:
        title += f' – {epoch_num}'
    plt.title(title)
    plt.xlabel('Rule frequency (# instances)')
    plt.ylabel('')
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()


def save_lore_rules_text(rules: list, epoch_num=None, mode: str = '', out_path=None) -> str:
    """Write per-instance LORE rules to a .txt file."""
    header = 'LORE Rules'
    if mode:
        header += f' [{mode}]'
    if epoch_num is not None:
        header += f' – {epoch_num}'
    lines = [header, '=' * len(header), '']

    def _fmt(subset, label):
        lines.append(f'=== {label} ({len(subset)} instances) ===')
        for i, r in subset:
            lines.append(f'[inst {i:02d}] pred={r["pred"]:.3f} label={r["label"]}')
            lines.append(f'  RULE: {r["rule_str"]}')
            for cr in r.get('crules_str', []):
                lines.append(f'  COUNTERFACTUAL: {cr}')
            lines.append('')

    pos = [(i, r) for i, r in enumerate(rules) if r['pred'] >= 0.5]
    neg = [(i, r) for i, r in enumerate(rules) if r['pred'] < 0.5]
    _fmt(pos, 'POSITIVE PREDICTIONS')
    _fmt(neg, 'NEGATIVE PREDICTIONS')

    text = "\n".join(lines)
    if out_path:
        with open(out_path, 'w') as f:
            f.write(text)
    return text

In [ ]:
from xailib.explainers.lore_explainer import LoreTabularExplainer
from collections import defaultdict
from typing import Dict, List, Optional


class LoreEngine:
    """
    LORE on raw (pre-FeatureSpace) tabular features.

    Pipeline per perturbation:
      raw features → FeatureSpace → head_model(cached_img_emb, encoded) → pred

    XAI-Lib auto-detects categoricals (via df._get_numeric_data) and samples
    from their empirical distribution — no one-hot contradictions.
    Rules are generated directly in the original feature space.
    """

    def __init__(
        self,
        feature_space,
        feat_cols: List[str],
        max_explain_samples: int = 12,
        positive_threshold: float = 0.5,
        num_lore_samples: int = 300,
        lore_config: Optional[Dict] = None,
    ):
        self.feature_space = feature_space
        self.feat_cols = feat_cols
        self.max_explain_samples = int(max_explain_samples)
        self.positive_threshold = float(positive_threshold)
        self.lore_config = lore_config or {
            'neigh_type': 'random',
            'size': int(num_lore_samples),
            'random_state': 42,
            'verbose': True,
        }
        self._image_encoder = None
        self._head_model = None

    # ── Model splitting (same as before — still needed for speed) ────────
    def _build_sub_models(self, model):
        if self._image_encoder is not None:
            return
        concat_layer = next(
            l for l in model.layers if isinstance(l, keras.layers.Concatenate)
        )
        img_branch = concat_layer.input[0]
        self._image_encoder = keras.Model(
            inputs=model.input["images"], outputs=img_branch, name="image_encoder",
        )
        emb_in = keras.Input(shape=(int(img_branch.shape[-1]),), name="cached_img_emb")
        feat_in = keras.Input(shape=(int(model.input["features"].shape[-1]),), name="cached_features")
        x = feat_in
        for name in ("dense", "dense_1", "dropout_1"):
            layer = model.get_layer(name)
            x = layer(x, training=False) if "dropout" in name else layer(x)
        merged = keras.layers.Concatenate()([emb_in, x])
        pred = model.get_layer("dense_2")(merged)
        self._head_model = keras.Model(inputs=[emb_in, feat_in], outputs=pred, name="head_model")

    # ── Predictor: raw features → FeatureSpace → head_model ─────────────
    def _make_predictor(self, image_embedding):
        head = self._head_model
        fs = self.feature_space
        emb_const = tf.constant(image_embedding[np.newaxis], dtype=tf.float32)

        def predict(X) -> np.ndarray:
            if isinstance(X, pd.DataFrame):
                feat_dict = {col: tf.constant(X[col].values) for col in X.columns
                             if col != 'target'}
            else:
                arr = np.asarray(X, dtype=np.float32)
                if arr.ndim == 1:
                    arr = arr.reshape(1, -1)
                feat_dict = {col: tf.constant(arr[:, i])
                             for i, col in enumerate(self.feat_cols) if col != 'target'}
            encoded = fs(feat_dict)
            n = encoded.shape[0]
            embs = tf.repeat(emb_const, n, axis=0)
            return head([embs, encoded], training=False).numpy()[:, 0]

        return predict

    # ── LORE wrappers ────────────────────────────────────────────────────
    def _build_explainer(self, predictor, background_df):
        class _BB:
            def __init__(self, fn, thr):
                self.fn, self.thr = fn, thr
            def predict(self, X):
                return (self.fn(X) >= self.thr).astype(int)
            def predict_proba(self, X):
                p = self.fn(X)
                return np.column_stack([1 - p, p])

        bb = _BB(predictor, self.positive_threshold)
        df_bg = background_df.copy().reset_index(drop=True)
        df_bg['target'] = bb.predict(df_bg).astype(np.int64)
        exp = LoreTabularExplainer(bb=bb)
        exp.fit(df_bg, 'target', self.lore_config)
        return exp

    def _explain_one(self, lore_exp, x_series):
        try:
            exp = lore_exp.explain(x_series.values)
            rule = exp.getRules()
            crules = exp.getCounterfactualRules()
            premise = rule.get('premise', [])

            rule_str = ' AND '.join(
                f"{p['att']} {p['op']} {p['thr']:.4g}" for p in premise
            )
            rule_features = [p['att'] for p in premise]
            crules_str = [
                ' AND '.join(f"{p['att']} {p['op']} {p['thr']:.4g}" for p in cr.get('premise', []))
                for cr in (crules or [])
            ]
            return {
                'rule_str': rule_str,
                'rule_features': rule_features,
                'crules_str': crules_str,
            }
        except Exception as e:
            return {'rule_str': f'[error: {e}]', 'rule_features': [], 'crules_str': []}

    # ── Public API ───────────────────────────────────────────────────────
    def explain(self, raw_df, images, labels, model, mode='context'):
        """
        raw_df: DataFrame with original feature columns
        images: numpy array of images.
        labels: numpy array of labels.
        """
        self._build_sub_models(model)

        n = len(raw_df)
        ex_idx = np.arange(n)
        if n > self.max_explain_samples:
            ex_idx = np.sort(np.random.choice(n, self.max_explain_samples, replace=False))

        ex_df = raw_df.iloc[ex_idx].reset_index(drop=True)
        ex_labels = labels[ex_idx].ravel()

        # Image embedding: computed once
        if mode == 'context':
            img = images.mean(axis=0, keepdims=True).astype(np.float32)
        elif mode == 'tabular_only':
            img = np.zeros((1, *images.shape[1:]), dtype=np.float32)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        image_embedding = self._image_encoder(img, training=False).numpy()[0]
        predictor = self._make_predictor(image_embedding)
        preds = predictor(ex_df)
        lore_exp = self._build_explainer(predictor, raw_df)

        freq_all, freq_pos, freq_neg = defaultdict(int), defaultdict(int), defaultdict(int)
        rules = []

        for i in range(len(ex_df)):
            result = self._explain_one(lore_exp, ex_df.iloc[i])
            result['pred'] = float(preds[i])
            result['label'] = int(ex_labels[i])
            rules.append(result)
            for feat in result['rule_features']:
                # XAI-Lib may one-hot encode internally (e.g. "sex=female")
                raw_feat = feat.split("=")[0] if "=" in feat else feat
                freq_all[raw_feat] += 1
                (freq_pos if preds[i] >= self.positive_threshold else freq_neg)[raw_feat] += 1

        stats = {
            'feature_freq_all': dict(freq_all),
            'feature_freq_pos': dict(freq_pos),
            'feature_freq_neg': dict(freq_neg),
            'explained_indices': ex_idx,
        }
        return stats, rules, preds

In [ ]:
from pathlib import Path


class LoreCallback(keras.callbacks.Callback):
    """
    Runs LoreEngine every N epochs on a frozen validation subset.
    """

    def __init__(
        self,
        data_source: tf.data.Dataset,
        raw_feature_df: pd.DataFrame,
        feat_cols: List[str],
        feature_space,
        base_dir: str,
        max_explain_samples: int = 12,
        num_lore_samples: int = 300,
        positive_threshold: float = 0.5,
        top_k: int = 10,
        every_n_epochs: int = 2,
        max_samples: int = 256,
        max_batches: Optional[int] = None,
        lore_config: Optional[Dict] = None,
        rng_seed: int = 0,
    ):
        super().__init__()
        self.data_source = data_source
        self.raw_feature_df = raw_feature_df  # pre-FeatureSpace validation DataFrame
        self.feat_cols = feat_cols
        self.top_k = int(top_k)
        self.every_n_epochs = max(1, int(every_n_epochs))
        self.max_samples = int(max_samples)
        self.max_batches = max_batches
        self.rng_seed = int(rng_seed)
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)

        self._images = None
        self._raw_df = None
        self._labels = None

        self._engine = LoreEngine(
            feature_space=feature_space,
            feat_cols=feat_cols,
            max_explain_samples=int(max_explain_samples),
            positive_threshold=float(positive_threshold),
            num_lore_samples=int(num_lore_samples),
            lore_config=lore_config,
        )

    def _freeze_batch(self, ds: tf.data.Dataset):
        """Collect images and labels from dataset, raw features from DataFrame."""
        imgs, labels = [], []
        n = 0
        for b_idx, (x, y) in enumerate(ds):
            if self.max_batches and b_idx >= self.max_batches:
                break
            remaining = self.max_samples - n
            if remaining <= 0:
                break
            imgs.append(x['images'].numpy()[:remaining])
            labels.append(y.numpy()[:remaining])
            n += imgs[-1].shape[0]

        if n == 0:
            raise RuntimeError('No samples collected for LORE.')
        self._images = np.concatenate(imgs)
        self._labels = np.concatenate(labels)
        # Take corresponding rows from raw DataFrame (same order, no shuffle in val)
        self._raw_df = self.raw_feature_df.iloc[:n][self.feat_cols].reset_index(drop=True)

    def on_train_begin(self, logs=None):
        self._freeze_batch(self.data_source)

    def on_epoch_end(self, epoch: int, logs=None):
        if epoch % self.every_n_epochs != 0 or self._images is None:
            return

        np.random.seed(self.rng_seed)
        epoch_name = f'epoch{epoch:03d}'
        epoch_dir = self.base_dir / epoch_name
        epoch_dir.mkdir(parents=True, exist_ok=True)

        for mode in ('context', 'tabular_only'):
            mode_dir = epoch_dir / mode
            mode_dir.mkdir(parents=True, exist_ok=True)

            stats, rules, preds = self._engine.explain(
                raw_df=self._raw_df,
                images=self._images,
                labels=self._labels,
                model=self.model,
                mode=mode,
            )

            plot_lore_feature_frequency(
                freq_all=stats['feature_freq_all'],
                freq_pos=stats['feature_freq_pos'],
                freq_neg=stats['feature_freq_neg'],
                k=self.top_k,
                epoch_num=epoch_name,
                mode=mode,
                out_path=str(mode_dir / f'lore_feature_freq_top{self.top_k:02d}_{mode}_{epoch_name}.png'),
            )

            save_lore_rules_text(
                rules=rules,
                epoch_num=epoch_name,
                mode=mode,
                out_path=str(mode_dir / f'lore_rules_{mode}_{epoch_name}.txt'),
            )

In [ ]:
XAI_LORE_PATH = os.path.join(MODEL_PATH, "xai", "lore_callback")
os.makedirs(XAI_LORE_PATH, exist_ok=True)

lore_cb = LoreCallback(
    data_source=validation_ds,
    raw_feature_df=validation_df,
    feat_cols=FEAT_COLS,
    feature_space=feature_space,
    base_dir=XAI_LORE_PATH,
    max_explain_samples=12,
    num_lore_samples=300,
    positive_threshold=0.5,
    top_k=10,
    every_n_epochs=2,
    max_samples=256,
    max_batches=10,
    rng_seed=0,
)
print(f"LORE output dir: {XAI_LORE_PATH}")

In [ ]:
model_fpath = os.path.join(MODEL_PATH, 'best_enhanced_model_test.keras')

ckpt_cb = keras.callbacks.ModelCheckpoint(
    model_fpath,              # Filepath where the model will be saved.
    monitor="val_auc",        # Metric to monitor (validation AUC in this case).
    save_best_only=True,      # Save only the model with the best performance.
    save_weights_only=False,  # Save the entire model (not just the weights).
    mode="max",               # The model with the maximum 'val_auc' will be saved.
)

In [ ]:
# cbs = [lr_cb, ckpt_cb, lore_cb]
# history = model.fit(
#     training_ds,
#     epochs=CFG.epochs,
#     callbacks=cbs,
#     validation_data=validation_ds,
#     verbose=CFG.verbose,
#     class_weight=class_weights,
# )


Debug

In [ ]:
engine = LoreEngine(
    feature_space=feature_space,
    feat_cols=FEAT_COLS,
    max_explain_samples=3,
    num_lore_samples=200,
)

# Get images + labels from encoded dataset, raw features from DataFrame
(x_batch, y_batch) = next(iter(validation_ds.take(1)))
images_small = x_batch['images'].numpy()[:64]
labels_small = y_batch.numpy()[:64]
raw_df_small = validation_df[FEAT_COLS].iloc[:64].reset_index(drop=True)

stats, rules, preds = engine.explain(
    raw_df=raw_df_small,
    images=images_small,
    labels=labels_small,
    model=model,
    mode='context',
)

print("Feature frequencies:", stats['feature_freq_all'])
for r in rules:
    print(f"pred={r['pred']:.3f} | {r['rule_str']}")

In [ ]:
# Manually trigger the callback hooks
lore_cb.set_model(model)          # inject the model (normally done by Keras)
lore_cb.on_train_begin()          # freezes the validation batch
lore_cb.on_epoch_end(epoch=0)     # runs LORE for epoch 0

for root, dirs, files in os.walk(XAI_LORE_PATH):
    for f in files:
        print(os.path.join(root, f))